# Lab 2 — Build simple RAG Q&A

Embeddings + cosine similarity.


In [1]:
from pathlib import Path

import pandas as pd

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
else:
    for p in [GH_ROOT, *GH_ROOT.parents]:
        if (p / "data" / "support-ops").is_dir():
            GH_ROOT = p
            break
SUPPORT_OPS = GH_ROOT / "data" / "support-ops"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

kb = (SUPPORT_OPS / "kb_snippets.txt").read_text(encoding="utf-8")
chunks = ["## " + c for c in kb.split("## ") if c.strip()]
query = "How do I handle VPN tunnel drop on IOS-XE?"
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode(chunks)
    q = model.encode([query])
except ImportError:
    print("SKIP embeddings model — using TF-IDF fallback")
    vec = TfidfVectorizer()
    emb = vec.fit_transform(chunks)
    q = vec.transform([query])
idx = cosine_similarity(q, emb).argmax()
print("Best chunk:\n", chunks[idx][:400])


SKIP embeddings model — using TF-IDF fallback
Best chunk:
 ## VPN tunnel drop (IOS-XE)
Verify IKE phase 1/2, check peer reachability, compare crypto map. Clear SA with clear crypto sa if policy allows.


